# Turkish Morph Retrieval — 300-family all-test evaluation

Bu deneme notebook'u mevcut **300 Codex family'nin tamamını test query'si** olarak değerlendirir.
Her family: `1 query + 1 positive + 8 hard negative + 2 easy negative`.
Orijinal 50 development + 250 sealed etiketi yalnız dağılım/provenance bilgisidir; ana metriklerin tamamı 300 query üzerinde hesaplanır.
Sonuçlar üretim kalitesini ve benchmark zorluğunu erken görmek içindir; 600-family nihai paper sonucu değildir.

V3.6 değerlendirme katmanları: veri/artefakt kontrolü → ucuz baseline'lar → dense encoder'lar → hard-negative
ayrımı → bootstrap CI → paired testler → fenomen/slice analizi → ablation → hata analizi → export.
Kontrollü ve full-corpus retrieval sonuçları ayrı cutoff setleriyle raporlanır.

## 0. Colab kullanımı

1. `Runtime > Change runtime type` ile GPU seçin (Qwen3-8B için A100 önerilir).
2. Aşağıdaki hücreleri sırayla çalıştırın.
3. Beş model A100'de sırayla yüklenir; her modelden sonra GPU belleği temizlenir.
4. A100 dışındaki runtime'larda Qwen3-8B'yi model tablosunda `enabled=False` yapabilirsiniz.
5. Çalışma yarıda kesilirse sonuç cache'i sayesinde tamamlanan modeller yeniden koşmaz.

In [ ]:
%pip -q install -U "sentence-transformers>=3.0,<6" "transformers>=4.48,<5" scikit-learn pandas matplotlib seaborn

In [ ]:
import gc, hashlib, json, random, shutil, subprocess, sys, time, traceback
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from IPython.display import display
pd.options.display.float_format = "{:.3f}".format

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

REPO_URL = "https://github.com/TR-morph-retrieval/turkish-morph-retrieval.git"
ROOT = Path("/content/turkish-morph-retrieval")

def run_git(command):
    result = subprocess.run(command, text=True, capture_output=True)
    if result.returncode != 0:
        raise RuntimeError("Public GitHub deposuna erisilemedi. Runtime internet baglantisini ve REPO_URL degerini kontrol edin.\nGit: " + result.stderr[-800:])

if (ROOT / "test/evaluation.py").exists():
    run_git(["git", "-C", str(ROOT), "pull", "--ff-only"])
else:
    if ROOT.exists(): shutil.rmtree(ROOT)  # yalnız yarım kalmış Colab clone'u
    run_git(["git", "clone", "--depth", "1", REPO_URL, str(ROOT)])
sys.path.insert(0, str(ROOT))

from sentence_transformers import SentenceTransformer
from test.evaluation import (
    EVALUATION_API_VERSION, FULL_CORPUS_RECALL_KS, ablate_items, approximate_randomization,
    artifact_baseline_summaries, bootstrap_ci, closed_qrels, evaluate_run,
    holm_adjust, load_items, mcnemar, paired_bootstrap, score_encoder, slice_summary,
)
assert EVALUATION_API_VERSION == "3.2", "Repo eski. Son dosyaları pull edip runtime'ı yeniden başlatın."
GIT_COMMIT = subprocess.check_output(["git", "-C", str(ROOT), "rev-parse", "HEAD"], text=True).strip()
print("repo:", ROOT)
print("git commit:", GIT_COMMIT)
print("device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

## 1. 300-family deneme ayarları

Bu notebook Git'teki ilk 300 doğrulanmış Codex shard'ını otomatik birleştirir.
Ara sonuçlarla model seçimi veya nihai paper iddiası dondurulmamalıdır.

In [ ]:
RUN_ID = "final_v39"
SHARD_DIR = ROOT / "test/data/final_shards"
EXPECTED_FAMILIES = 300
BATCH_SIZE = 16
FULL_RUN_DEPTH = 50             # Full-corpus Recall@50 için
N_BOOT = 10_000
USE_CACHE = True
OUTPUT_NAME = "morph_eval_300_alltest_v390"
OUTPUT_DIR = Path("/content") / OUTPUT_NAME if Path("/content").exists() else ROOT / "test/results" / OUTPUT_NAME
CACHE_DIR = OUTPUT_DIR / "cache"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
SHARD_FILES = sorted(SHARD_DIR.glob("codex_*.jsonl"))
assert SHARD_FILES, f"Codex shard bulunamadı: {SHARD_DIR}"
EVAL_ITEMS = []
for shard in SHARD_FILES:
    EVAL_ITEMS.extend(json.loads(line) for line in shard.read_text(encoding="utf-8").splitlines() if line.strip())
assert len(EVAL_ITEMS) == EXPECTED_FAMILIES, (
    f"{EXPECTED_FAMILIES} family bekleniyordu, {len(EVAL_ITEMS)} bulundu. Repo main'i yeniden pull edin."
)
ORIGINAL_DEV = [item for item in EVAL_ITEMS if item.get("target_split") == "development"]
ORIGINAL_SEALED = [item for item in EVAL_ITEMS if item.get("target_split") == "sealed_test"]
assert (len(ORIGINAL_DEV), len(ORIGINAL_SEALED)) == (50, 250), (
    f"Provenance'ta 50/250 split bekleniyordu: {len(ORIGINAL_DEV)}/{len(ORIGINAL_SEALED)}"
)
# Ana encoder, CI, slice, full-corpus ve query-blind OOF metriklerinde 300 family'sinin tamamı testtir.
DATA_FILE = SHARD_DIR
DATA_NOTICE = "ALL-TEST DENEME: 300 Codex family ana metriklerde birlikte değerlendirilir; nihai paper sonucu değildir."
DATA_BYTES = b"".join(path.read_bytes() for path in SHARD_FILES)
DATA_SHA256 = hashlib.sha256(DATA_BYTES).hexdigest()
print(DATA_NOTICE)
print(f"family={len(EVAL_ITEMS)} | candidate={sum(len(x['candidates']) for x in EVAL_ITEMS)}")
print("data sha256:", DATA_SHA256)

## 2. Veri bütünlüğü ve dağılım

In [ ]:
integrity_rows = []
for item in EVAL_ITEMS:
    roles = Counter(c["role"] for c in item["candidates"])
    ids = [c["id"] for c in item["candidates"]]
    integrity_rows.append({
        "family_id": item["family_id"],
        "candidate_n": len(ids),
        "positive_n": roles["positive"],
        "hard_n": roles["hard_negative"],
        "easy_n": roles["easy_negative"],
        "unique_ids": len(ids) == len(set(ids)),
        "gold_exists": item["gold_id"] in ids,
    })
integrity = pd.DataFrame(integrity_rows)
display(integrity)
assert integrity[["unique_ids", "gold_exists"]].all().all()
assert (integrity[["candidate_n", "positive_n", "hard_n", "easy_n"]] == [11, 1, 8, 2]).all().all()
family_ids = [item["family_id"] for item in EVAL_ITEMS]
corpus_ids = [candidate["id"] for item in EVAL_ITEMS for candidate in item["candidates"]]
assert len(family_ids) == len(set(family_ids)), "Tekrarlanan family_id var"
assert len(corpus_ids) == len(set(corpus_ids)), "Family'ler arasında tekrarlanan candidate id var"
assert Counter(item.get("generator_id") for item in EVAL_ITEMS) == {"generator_a": 300}
print(f"✓ 1/8/2 yapı, 300 benzersiz family ve Codex generator kaynağı doğrulandı; strict={sum(bool(item.get('strict_minimal_pair')) for item in EVAL_ITEMS)}.")

In [ ]:
fields = [
    "split", "query_sentence_count", "passage_sentence_count", "layer", "objective",
    "generalization_bucket", "macro_phenomenon", "target_feature", "family_mode",
    "query_expression", "query_gold_lexical_band", "domain", "register",
    "strict_minimal_pair", "generator_id",
]
for field in fields:
    values = [item.get("split", item.get("target_split")) if field == "split" else item.get(field) for item in EVAL_ITEMS]
    counts = pd.Series([str(value) for value in values]).value_counts().rename("n").to_frame()
    counts["ratio"] = counts["n"] / len(EVAL_ITEMS)
    print("\n", field)
    display(counts)

In [ ]:
from test.taxonomy import FEATURES

def distribution_frame(field, order=None):
    values = [item.get(field) for item in EVAL_ITEMS]
    counts = pd.Series(values).value_counts()
    if order is not None:
        counts = counts.reindex(order, fill_value=0)
    frame = counts.rename_axis("value").reset_index(name="n")
    frame["ratio"] = frame["n"] / len(EVAL_ITEMS)
    frame["field"] = field
    return frame

overview_fields = [
    "macro_phenomenon", "objective", "family_mode",
    "domain", "register", "generalization_bucket",
]
fig, axes = plt.subplots(3, 2, figsize=(15, 17))
for ax, field in zip(axes.flat, overview_fields):
    frame = distribution_frame(field).sort_values("n")
    sns.barplot(data=frame, x="n", y="value", ax=ax, color="#4C78A8")
    ax.set_title(field.replace("_", " "))
    ax.set(xlabel="family sayısı", ylabel="")
    ax.bar_label(ax.containers[0], padding=3, fontsize=9)
fig.suptitle("300-family test dağılımı", fontsize=16, y=1.01)
plt.tight_layout()
fig.savefig(OUTPUT_DIR / "distribution_overview.png", dpi=160, bbox_inches="tight")
plt.show()

design_fields = [
    ("query_sentence_count", [1, 2]),
    ("passage_sentence_count", [1, 2, 3, 4]),
    ("query_expression", ["morph_explicit", "semantic_paraphrase"]),
    ("query_gold_lexical_band", ["low", "medium", "high"]),
]
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
for ax, (field, order) in zip(axes.flat, design_fields):
    frame = distribution_frame(field, order)
    sns.barplot(data=frame, x="value", y="n", ax=ax, color="#59A14F")
    ax.set_title(field.replace("_", " "))
    ax.set(xlabel="", ylabel="family sayısı")
    ax.tick_params(axis="x", rotation=20)
    ax.bar_label(ax.containers[0], padding=3, fontsize=9)
plt.tight_layout()
fig.savefig(OUTPUT_DIR / "distribution_design.png", dpi=160, bbox_inches="tight")
plt.show()

all_feature_keys = [feature.key for feature in FEATURES]
feature_frame = distribution_frame("target_feature", all_feature_keys).sort_values("n")
fig, ax = plt.subplots(figsize=(12, 19))
sns.barplot(data=feature_frame, x="n", y="value", ax=ax, color="#F28E2B")
ax.set(title="76 fenomenin ilk 300 family içindeki kapsamı", xlabel="family sayısı", ylabel="")
ax.bar_label(ax.containers[0], padding=2, fontsize=8)
plt.tight_layout()
fig.savefig(OUTPUT_DIR / "distribution_features.png", dpi=160, bbox_inches="tight")
plt.show()

missing_features = feature_frame.loc[feature_frame.n.eq(0), "value"].tolist()
priority_n = sum(bool(item.get("qc", {}).get("human_review_priority")) for item in EVAL_ITEMS)
print(f"Fenomen kapsamı: {(feature_frame.n > 0).sum()}/76; henüz görünmeyenler: {missing_features}")
print(f"Final insan incelemesinde öncelikli: {priority_n}/300 ({priority_n / 300:.1%}); bu etiket otomatik red değildir.")

distribution_long = pd.concat(
    [distribution_frame(field) for field in overview_fields]
    + [distribution_frame(field, order) for field, order in design_fields]
    + [feature_frame], ignore_index=True
)
distribution_long.to_csv(OUTPUT_DIR / "distribution_summary.csv", index=False)

## 3. Metrikler nasıl okunmalı?

**Ana morfoloji metrikleri:**

- `pairwise_hard_accuracy`: gold–hard çiftlerinin ne kadarında gold daha yüksek skor aldı? Chance %50.
- `pairwise_morph_hard_accuracy` / `pairwise_semantic_hard_accuracy`: iki zorluk kaynağını ayırır.
- `all_hard_family_consistency`: gold aynı family'deki sekiz hard'ın tamamını geçti mi?
- `contrast_consistency`: gold, minimal morfolojik negatifi geçti mi?
- `hardest_hard_margin`: gold skoru − en yüksek hard skoru. Pozitif değer iyi.

**11 adaylık kontrollü metrikler:** `Recall@1/3`, `MRR@10`, `nDCG@10`.
**3.300-belge full-corpus metrikleri:** `Recall@1/3/10/50`, `MRR@10`, `nDCG@10`.

## 4. Query-blind / ucuz artefakt baseline'ları

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import KFold

def candidate_only_oof(items, n_splits=5):
    """Family-level OOF query-blind classifier; hiçbir family kendi fold'unda eğitime girmez."""
    run = {}
    folds = KFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    indices = np.arange(len(items))
    for train_idx, test_idx in folds.split(indices):
        train_items = [items[index] for index in train_idx]
        train_texts = [candidate["text"] for item in train_items for candidate in item["candidates"]]
        train_labels = [int(candidate["id"] == item["gold_id"]) for item in train_items for candidate in item["candidates"]]
        vectorizer = TfidfVectorizer(analyzer="char", ngram_range=(3, 5), min_df=2, max_features=50_000)
        classifier = LogisticRegression(max_iter=2_000, class_weight="balanced", random_state=SEED)
        classifier.fit(vectorizer.fit_transform(train_texts), train_labels)
        for index in test_idx:
            item = items[index]
            texts = [candidate["text"] for candidate in item["candidates"]]
            scores = classifier.predict_proba(vectorizer.transform(texts))[:, 1]
            order = sorted(range(len(texts)), key=lambda i: (-scores[i], item["candidates"][i]["id"]))
            run[item["family_id"]] = [item["candidates"][i]["id"] for i in order]
    summary, per_query = evaluate_run(closed_qrels(items), run)
    return {"summary": summary, "per_query": per_query, "run": run}

qrels = closed_qrels(EVAL_ITEMS)
# Position baseline sabit ilk sırayı seçer; test gold konumlarından öğrenmez.
artifact_summaries = artifact_baseline_summaries(EVAL_ITEMS, learned_position=0)
try:
    artifact_summaries["candidate_only_char_tfidf_oof"] = candidate_only_oof(EVAL_ITEMS)["summary"]
except Exception as exc:
    print("candidate-only OOF classifier atlandı:", exc)

ARTIFACT_DF = pd.DataFrame(artifact_summaries).T.sort_values("recall@1", ascending=False)
display(ARTIFACT_DF[["recall@1", "recall@3", "mrr@10", "ndcg@10", "mean_rank"]].round(3))
print("closed R@1 chance =", round(1 / 11, 4))

In [ ]:
ax = ARTIFACT_DF["recall@1"].sort_values().plot.barh(figsize=(8, 4), color="#7a9cc6")
ax.axvline(1 / 11, color="black", linestyle="--", label="chance 1/11")
ax.set(title="Ucuz baseline Recall@1", xlabel="Recall@1", ylabel="")
ax.legend(); plt.tight_layout(); plt.show()

## 5. Encoder kayıt defteri

Varsayılanlar A100 icin mevcut proje listesindeki 5 encoder'dir. Bir model yüklenemezse deney durmaz; hata kaydedilir.
Prefix'ler model ailesinin retrieval biçimine göre ayrı tutulur. Qwen3-8B belleği yüksek olduğu
için A100/L4 disi runtime'larda kapatabilirsiniz. Aynı notebook'u tekrar çalıştırırken model veya prefix değişirse cache anahtarı da değişir.

In [ ]:
INSTRUCT = "Instruct: Given a web search query, retrieve relevant passages that answer the query\nQuery:"
MODEL_SPECS = [
    {"name": "e5-large", "repo": "intfloat/multilingual-e5-large", "query_prefix": "query: ", "document_prefix": "passage: ", "enabled": True},
    {"name": "bge-m3", "repo": "BAAI/bge-m3", "query_prefix": "", "document_prefix": "", "enabled": True},
    {"name": "modernbert-tr", "repo": "ytu-ce-cosmos/modernbert-tr-embed", "query_prefix": INSTRUCT, "document_prefix": "", "enabled": True},
    {"name": "trmteb-ft-110m", "repo": "trmteb/turkish-embedding-model-fine-tuned", "query_prefix": "", "document_prefix": "", "enabled": True},
    {"name": "qwen3-8b", "repo": "Qwen/Qwen3-Embedding-8B", "query_prefix": INSTRUCT, "document_prefix": "", "enabled": True, "dtype": "float16"},
]
display(pd.DataFrame(MODEL_SPECS)[["name", "repo", "enabled", "query_prefix", "document_prefix"]])

In [ ]:
EVAL_CODE_SHA256 = hashlib.sha256((ROOT / "test/evaluation.py").read_bytes()).hexdigest()

def cache_path(spec):
    payload = json.dumps({"spec": spec, "data": DATA_SHA256, "eval": EVAL_CODE_SHA256}, sort_keys=True)
    key = hashlib.sha256(payload.encode()).hexdigest()[:12]
    return CACHE_DIR / f"{spec['name']}-{key}.json"

def load_model(spec):
    kwargs = {"trust_remote_code": True}
    if spec.get("dtype") and torch.cuda.is_available():
        kwargs["model_kwargs"] = {"torch_dtype": getattr(torch, spec["dtype"])}
    model = SentenceTransformer(spec["repo"], **kwargs)
    try:
        model.default_prompt_name = None
    except Exception:
        pass
    return model

def score_with_backoff(model, items, spec, include_full_run=False):
    batch_size = BATCH_SIZE
    while True:
        try:
            result = score_encoder(
                model, items, spec.get("query_prefix", ""), spec.get("document_prefix", ""),
                full_corpus=False, batch_size=batch_size, include_full_run=include_full_run,
                full_run_depth=FULL_RUN_DEPTH if include_full_run else None,
            )
            return result, batch_size
        except torch.cuda.OutOfMemoryError:
            if batch_size == 1:
                raise
            batch_size = max(1, batch_size // 2)
            gc.collect()
            torch.cuda.empty_cache()
            print(f"CUDA OOM; batch_size={batch_size} ile yeniden deneniyor")

def run_one_model(spec):
    path = cache_path(spec)
    if USE_CACHE and path.exists():
        print(spec["name"], "cache'ten yüklendi")
        return json.loads(path.read_text()), {"model": spec["name"], "cached": True}
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
    started = time.perf_counter()
    model = load_model(spec)
    load_seconds = time.perf_counter() - started
    dimension = model.get_sentence_embedding_dimension()
    max_seq_length = model.max_seq_length
    try:
        resolved_revision = model[0].auto_model.config._commit_hash
    except Exception:
        resolved_revision = None
    score_started = time.perf_counter()
    result, effective_batch_size = score_with_backoff(model, EVAL_ITEMS, spec, include_full_run=True)
    score_seconds = time.perf_counter() - score_started
    peak_gb = torch.cuda.max_memory_allocated() / 1e9 if torch.cuda.is_available() else 0.0
    path.write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
    del model
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return result, {
        "model": spec["name"], "cached": False, "dimension": dimension,
        "max_seq_length": max_seq_length, "resolved_revision": resolved_revision,
        "effective_batch_size": effective_batch_size, "load_seconds": load_seconds,
        "score_seconds": score_seconds, "peak_gpu_gb": peak_gb,
    }

## 6. Encoder'ları çalıştır

In [ ]:
RESULTS, RUNTIME_ROWS, MODEL_ERRORS = {}, [], []
for spec in [spec for spec in MODEL_SPECS if spec["enabled"]]:
    print("\n===", spec["name"], "===")
    try:
        RESULTS[spec["name"]], runtime = run_one_model(spec)
        RUNTIME_ROWS.append(runtime)
    except Exception as exc:
        MODEL_ERRORS.append({"model": spec["name"], "error": repr(exc), "traceback": traceback.format_exc()})
        print("ATLANDI:", repr(exc))
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()
assert RESULTS, "Hiçbir model tamamlanmadı. MODEL_ERRORS çıktısını inceleyin."
display(pd.DataFrame(RUNTIME_ROWS))
if MODEL_ERRORS: display(pd.DataFrame(MODEL_ERRORS)[["model", "error"]])

## 7. Ana sonuç tablosu

In [ ]:
SUMMARY_DF = pd.DataFrame({name: result["summary"] for name, result in RESULTS.items()}).T
primary = [
    "hard_only_mrr@10", "pairwise_hard_accuracy",
    "pairwise_morph_hard_accuracy", "pairwise_semantic_hard_accuracy",
    "all_hard_family_consistency", "contrast_consistency",
    "hardest_hard_margin", "recall@1", "recall@3", "mrr@10", "ndcg@10",
    "mean_rank", "mean_hard_rank",
]
display(SUMMARY_DF[primary].sort_values("recall@1", ascending=False).round(3))
print("hard-only R@1 chance:", round(1 / 9, 4), "| pairwise chance: 0.5 | closed R@1 chance:", round(1 / 11, 4))

## 8. Query-level bootstrap %95 güven aralıkları

In [ ]:
CI_METRICS = ["pairwise_hard_accuracy", "pairwise_morph_hard_accuracy", "pairwise_semantic_hard_accuracy", "all_hard_family_consistency", "recall@1", "mrr@10", "ndcg@10"]
ci_rows = []
for model_name, result in RESULTS.items():
    for metric in CI_METRICS:
        values = [row[metric] for row in result["per_query"] if row.get(metric) is not None]
        low, high = bootstrap_ci(values, n_boot=N_BOOT, seed=SEED)
        ci_rows.append({"model": model_name, "metric": metric, "mean": np.mean(values), "ci_low": low, "ci_high": high, "n": len(values)})
CI_DF = pd.DataFrame(ci_rows)
display(CI_DF.round(3))
print(f"Ara değerlendirme CI'ları {len(EVAL_ITEMS)} query üzerinde query-level bootstrap ile hesaplandı.")

In [ ]:
plot_df = CI_DF[CI_DF.metric.isin(["pairwise_hard_accuracy", "recall@1"])].copy()
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for ax, metric in zip(axes, plot_df.metric.unique()):
    part = plot_df[plot_df.metric == metric].sort_values("mean")
    ax.errorbar(part["mean"], part["model"], xerr=[part["mean"] - part["ci_low"], part["ci_high"] - part["mean"]], fmt="o", capsize=3)
    ax.set_title(metric); ax.set_xlim(-0.03, 1.03); ax.grid(axis="x", alpha=.25)
plt.tight_layout(); plt.show()

## 9. Hard-negative subtype analizi

In [ ]:
subtype_rows = []
for model_name, result in RESULTS.items():
    for item in EVAL_ITEMS:
        scores = result["scores"][item["family_id"]]
        gold_score = scores[item["gold_id"]]
        for candidate in item["candidates"]:
            if candidate["role"] != "hard_negative": continue
            margin = gold_score - scores[candidate["id"]]
            subtype_rows.append({
                "model": model_name, "family_id": item["family_id"], "subtype": candidate["subtype"],
                "target_feature": item["target_feature"], "margin": margin,
                "gold_wins": 1.0 if margin > 0 else 0.5 if margin == 0 else 0.0,
            })
SUBTYPE_PAIRS_DF = pd.DataFrame(subtype_rows)
SUBTYPE_DF = (SUBTYPE_PAIRS_DF.groupby(["model", "subtype"])
              .agg(n=("gold_wins", "size"), accuracy=("gold_wins", "mean"), mean_margin=("margin", "mean"))
              .reset_index())
display(SUBTYPE_DF.sort_values(["model", "accuracy"]).round(3))
print("n küçük subtype satırlarını yalnız tanısal okuyun; inferential sonuç değildir.")

In [ ]:
heat = SUBTYPE_DF.pivot(index="subtype", columns="model", values="accuracy")
plt.figure(figsize=(max(7, 1.6 * len(RESULTS)), max(5, .45 * len(heat))))
sns.heatmap(heat, annot=True, fmt=".2f", cmap="RdYlGn", vmin=0, vmax=1)
plt.title("Gold'un hard negatifi geçme oranı"); plt.tight_layout(); plt.show()

## 10. Slice sonuçları

In [ ]:
slice_rows = []
for model_name, result in RESULTS.items():
    for metric in ["pairwise_hard_accuracy", "pairwise_morph_hard_accuracy", "pairwise_semantic_hard_accuracy", "recall@1"]:
        nested = slice_summary(result["per_query"], EVAL_ITEMS, metric=metric)
        for field, groups in nested.items():
            for value, stats in groups.items():
                slice_rows.append({"model": model_name, "metric": metric, "field": field, "value": value, **stats})
SLICE_DF = pd.DataFrame(slice_rows)
display(SLICE_DF.sort_values(["metric", "field", "model", "value"]).round(3))
print("n<5 dilimler yalnız hata bulma amaçlıdır; ayrı paper iddiası kurmayın.")

## 11. Hata analizi: model neyi birinci getirdi?

In [ ]:
item_by_id = {item["family_id"]: item for item in EVAL_ITEMS}
error_rows = []
for model_name, result in RESULTS.items():
    per_query = {row["query_id"]: row for row in result["per_query"]}
    for query_id, ranking in result["run"].items():
        item = item_by_id[query_id]
        candidates = {candidate["id"]: candidate for candidate in item["candidates"]}
        top = candidates[ranking[0]]
        row = per_query[query_id]
        error_rows.append({
            "model": model_name, "family_id": query_id, "correct@1": int(ranking[0] == item["gold_id"]),
            "gold_rank": row["rank"], "hard_rank": row["hard_rank"],
            "hardest_hard_margin": row["hardest_hard_margin"], "target_feature": item["target_feature"],
            "layer": item["layer"], "predicted_role": top["role"], "predicted_subtype": top["subtype"],
            "query": item["query"], "predicted_text": top["text"],
            "gold_text": candidates[item["gold_id"]]["text"],
        })
ERRORS_DF = pd.DataFrame(error_rows)
display(ERRORS_DF[ERRORS_DF["correct@1"] == 0].sort_values(["model", "gold_rank"]).reset_index(drop=True))

## 12. Paired model karşılaştırmaları

In [ ]:
def aligned_values(result, metric):
    return {row["query_id"]: float(row[metric]) for row in result["per_query"]}

comparison_rows, raw_p = [], {}
names = list(RESULTS)
for i, left_name in enumerate(names):
    for right_name in names[i + 1:]:
        for metric in ["pairwise_hard_accuracy", "recall@1", "ndcg@10"]:
            left_map, right_map = aligned_values(RESULTS[left_name], metric), aligned_values(RESULTS[right_name], metric)
            ids = sorted(set(left_map) & set(right_map))
            left, right = [left_map[x] for x in ids], [right_map[x] for x in ids]
            boot = paired_bootstrap(left, right, n_boot=N_BOOT, seed=SEED)
            p = approximate_randomization(left, right, n_iter=N_BOOT, seed=SEED)
            key = f"{left_name}__{right_name}__{metric}"
            raw_p[key] = p
            row = {"comparison": f"{left_name} - {right_name}", "metric": metric, "p_randomization": p, **boot}
            if metric == "recall@1":
                row.update({f"mcnemar_{k}": v for k, v in mcnemar(left, right).items()})
            comparison_rows.append(row)
adjusted = holm_adjust(raw_p) if raw_p else {}
for row in comparison_rows:
    left, right = row["comparison"].split(" - ")
    row["p_holm"] = adjusted[f"{left}__{right}__{row['metric']}"]
COMPARISONS_DF = pd.DataFrame(comparison_rows)
display(COMPARISONS_DF.round(3))
print("Effect size, güven aralığı ve Holm-düzeltilmiş p-değerlerini birlikte okuyun.")

## 13. Kritik sözcük / prefix-5 ablation

`prefix5`, gerçek Türkçe kök/lemma analizi değildir; ek bilgisini azaltan ucuz bir kontrol deneyidir.

In [ ]:
FOCUS = SUMMARY_DF["pairwise_hard_accuracy"].idxmax()
focus_spec = next(spec for spec in MODEL_SPECS if spec["name"] == FOCUS)
focus_model = load_model(focus_spec)
ablation_sets = {
    "original": EVAL_ITEMS,
    "critical_deleted": ablate_items(EVAL_ITEMS, "critical_deleted"),
    "prefix5": ablate_items(EVAL_ITEMS, "prefix5"),
}
ABLATION_RESULTS = {}
for ablation_name, items in ablation_sets.items():
    ABLATION_RESULTS[ablation_name], _ = score_with_backoff(focus_model, items, focus_spec)
del focus_model; gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()
ABLATION_DF = pd.DataFrame({name: result["summary"] for name, result in ABLATION_RESULTS.items()}).T
ablation_metrics = ["pairwise_hard_accuracy", "contrast_consistency", "recall@1", "mrr@10", "ndcg@10"]
display(ABLATION_DF[ablation_metrics].round(3))
print("focus model:", FOCUS)

## 14. Full-corpus retrieval

Bu katman **her zaman** ortak corpus sıralamasını üretir:

- Ara analizde 300 query, bütün 3.300 aday içinde aranır.
- Her query için tasarım gereği tek gold vardır; diğer family'ler farklı semantic frame taşır.
- Cross-family exact/fuzzy duplicate ve frame kontrollerinden geçen diğer belgeler nonrelevant kabul edilir.
- `Recall@1/3/10/50`, `MRR@10` ve `nDCG@10` bütün corpus sıralaması üzerinde hesaplanır.

In [ ]:
# Full-corpus retrieval: 300 ara-değerlendirme query'si bütün 3.300 belgeyi sıralar.
full_corpus = {}
full_per_query_frames = []
for model_name, result in RESULTS.items():
    if "full_run" not in result:
        raise RuntimeError(f"{model_name} cache'i full_run içermiyor. Cache'i silip modeli yeniden çalıştırın.")
    summary, rows = evaluate_run(closed_qrels(EVAL_ITEMS), result["full_run"], recall_ks=FULL_CORPUS_RECALL_KS)
    full_corpus[model_name] = summary
    frame = pd.DataFrame(rows)
    frame.insert(0, "model", model_name)
    full_per_query_frames.append(frame)

FULL_CORPUS_DF = pd.DataFrame(full_corpus).T
FULL_CORPUS_PER_QUERY_DF = pd.concat(full_per_query_frames, ignore_index=True)
display(FULL_CORPUS_DF[[
    "recall@1", "recall@3", "recall@10", "recall@50", "mrr@10", "ndcg@10"
]].sort_values("recall@10", ascending=False).round(3))
print(f"Full corpus: {len(EVAL_ITEMS)} query, {sum(len(x['candidates']) for x in EVAL_ITEMS):,} ortak belge, query başına tek gold.")

## 15. Sonuçları dışa aktar

In [ ]:
SUMMARY_DF.to_csv(OUTPUT_DIR / "encoder_summary.csv", index_label="model")
FULL_CORPUS_DF.to_csv(OUTPUT_DIR / "full_corpus_retrieval.csv", index_label="model")
FULL_CORPUS_PER_QUERY_DF.to_csv(OUTPUT_DIR / "full_corpus_retrieval_per_query.csv", index=False)
ARTIFACT_DF.to_csv(OUTPUT_DIR / "artifact_baselines.csv", index_label="baseline")
CI_DF.to_csv(OUTPUT_DIR / "bootstrap_ci.csv", index=False)
SUBTYPE_DF.to_csv(OUTPUT_DIR / "hard_subtype_results.csv", index=False)
SLICE_DF.to_csv(OUTPUT_DIR / "slice_results.csv", index=False)
ERRORS_DF.to_csv(OUTPUT_DIR / "error_analysis.csv", index=False)
COMPARISONS_DF.to_csv(OUTPUT_DIR / "paired_comparisons.csv", index=False)
ABLATION_DF.to_csv(OUTPUT_DIR / "ablations.csv", index_label="ablation")
pd.DataFrame(RUNTIME_ROWS).to_csv(OUTPUT_DIR / "runtime.csv", index=False)
(OUTPUT_DIR / "model_errors.json").write_text(json.dumps(MODEL_ERRORS, ensure_ascii=False, indent=2), encoding="utf-8")
metadata = {
    "dataset_mode": "midpoint_300_all_test_codex", "dataset_file": str(DATA_FILE), "dataset_sha256": DATA_SHA256,
    "evaluation_code_sha256": EVAL_CODE_SHA256, "evaluation_api": EVALUATION_API_VERSION,
    "git_commit": GIT_COMMIT, "full_run_depth": FULL_RUN_DEPTH,
    "family_count": len(EVAL_ITEMS), "seed": SEED, "bootstrap_draws": N_BOOT,
    "model_specs": MODEL_SPECS, "notice": DATA_NOTICE,
}
(OUTPUT_DIR / "run_metadata.json").write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding="utf-8")
archive = shutil.make_archive(str(OUTPUT_DIR), "zip", root_dir=OUTPUT_DIR)
print("çıktı:", OUTPUT_DIR)
print("zip:", archive)
try:
    from google.colab import files
    files.download(archive)
except ImportError:
    pass

## 16. Ara sonucu yorumlama sırası

1. Önce ucuz baseline'ların yüksek olup olmadığına bakın; yüksekse veri artefaktı vardır.
2. Encoder'larda önce `recall@1`, `pairwise_hard_accuracy` ve margin'leri okuyun.
3. Hangi hard subtype'ların sürekli kaybedildiğini inceleyin.
4. Kritik sözcük silinince skorun düşmesi morfolojik sinyale duyarlılıkla uyumludur; tek başına nedensellik kanıtı değildir.
5. Ana metriklerin tümü 300 test query üzerinde hesaplanır; `target_split` bu notebook'ta yalnız provenance alanıdır.
6. Bu 300-family koşusunu pipeline ve zorluk kontrolü olarak kullanın; nihai model/ayar kararını dondurmayın.
7. `full_corpus_retrieval.csv` bütün 3.300 belge üzerindeki retrieval sonucudur; kontrollü tabloyla birlikte okuyun.
8. Fine-tuning seed varyansı frozen encoder baseline'ına uygulanmaz; training aşamasında en az üç seed ayrıca raporlanmalı.